# Lab 4 exercise: deployed digital twin

My digital twin is available at [Andras Nemes Career Conversations](https://huggingface.co/spaces/andras-nemes1979/andras-nemes-career-conversations).

Note: Hugging Face disables the site after a period of inactivity. I do not know the exact inactivity period, so the Space may need to be restarted before use.

I have also improved the resources available to the digital twin. The `twin` folder next to this notebook contains my CV (`cv.pdf`), my LinkedIn profile (`linkedin.pdf`) and a broader `sources.txt` file with much more information about me: a personal summary, an elevator pitch and other source material. All three files are loaded into the LLM's context by the [digital_twin_app_with_evaluator](digital_twin_app_with_evaluator/app.py) app, which follows the modular structure of the updated course twin (`app.py`, `tools.py`, `context.py`, `styles.py`) and also brings in the evaluator pattern from the Day 4 exercise. This partially addresses the second exercise task.

I did not get the opportunity to fully implement RAG, but I prepared a concrete [RAG implementation plan](RAG_implementation_plan.MD) for converting the current approach into a proper retrieval pipeline.

## Demo: adding MySQL tools

This example adds two narrow tools: one retrieves matching FAQ entries and one saves an approved question and answer. The functions use parameterized queries instead of allowing the model to run arbitrary SQL. The existing `handle_tool_calls` function can discover them by name through `globals()`.

Install `mysql-connector-python` and set `MYSQL_HOST`, `MYSQL_PORT`, `MYSQL_DATABASE`, `MYSQL_USER`, and `MYSQL_PASSWORD` in `.env` before running the code.

In [ ]:
import os
import mysql.connector


def get_mysql_connection():
    return mysql.connector.connect(
        host=os.getenv("MYSQL_HOST", "localhost"),
        port=int(os.getenv("MYSQL_PORT", "3306")),
        database=os.getenv("MYSQL_DATABASE"),
        user=os.getenv("MYSQL_USER"),
        password=os.getenv("MYSQL_PASSWORD"),
    )


def create_faq_table():
    connection = get_mysql_connection()
    cursor = connection.cursor()
    try:
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS career_faq (
                id BIGINT AUTO_INCREMENT PRIMARY KEY,
                question VARCHAR(500) NOT NULL UNIQUE,
                answer TEXT NOT NULL,
                updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                    ON UPDATE CURRENT_TIMESTAMP
            )
        """)
        connection.commit()
    finally:
        cursor.close()
        connection.close()


def lookup_faq(question, limit=3):
    safe_limit = max(1, min(int(limit), 10))
    search_text = f"%{question}%"
    connection = get_mysql_connection()
    cursor = connection.cursor(dictionary=True)
    try:
        cursor.execute(
            f"""
            SELECT question, answer
            FROM career_faq
            WHERE question LIKE %s OR answer LIKE %s
            ORDER BY updated_at DESC
            LIMIT {safe_limit}
            """,
            (search_text, search_text),
        )
        return {"matches": cursor.fetchall()}
    finally:
        cursor.close()
        connection.close()


def save_faq(question, answer):
    connection = get_mysql_connection()
    cursor = connection.cursor()
    try:
        cursor.execute(
            """
            INSERT INTO career_faq (question, answer)
            VALUES (%s, %s)
            ON DUPLICATE KEY UPDATE
                answer = %s, updated_at = CURRENT_TIMESTAMP
            """,
            (question, answer, answer),
        )
        connection.commit()
        return {"saved": True}
    finally:
        cursor.close()
        connection.close()


# Run this once after configuring the database.
# create_faq_table()

In [ ]:
lookup_faq_json = {
    "name": "lookup_faq",
    "description": "Search the career FAQ before answering a question about Andras",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string"},
            "limit": {"type": "integer", "minimum": 1, "maximum": 10},
        },
        "required": ["question"],
        "additionalProperties": False,
    },
}

save_faq_json = {
    "name": "save_faq",
    "description": "Save an approved career question and answer for future conversations",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string"},
            "answer": {"type": "string"},
        },
        "required": ["question", "answer"],
        "additionalProperties": False,
    },
}

mysql_tools = [
    {"type": "function", "function": lookup_faq_json},
    {"type": "function", "function": save_faq_json},
]

# Add these definitions to the tools list from the digital twin app.
tools.extend(mysql_tools)

## Adding the evaluator pattern

In the previous lab, the evaluator checked the completed reply before it was returned. An acceptable reply passed through unchanged; a rejected reply was generated once more using the evaluator's feedback. Here the evaluator is placed after the digital twin finishes its tool-call loop and before `chat` returns the final text. The retry does not receive tools, which prevents notifications or database writes from running twice.

In [ ]:
from pydantic import BaseModel


class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


google_api_key = os.getenv("GOOGLE_API_KEY")
if not google_api_key:
    raise RuntimeError("GOOGLE_API_KEY is not set in .env")

gemini = OpenAI(
    api_key=google_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

evaluator_system_prompt = f"""You evaluate replies produced by a digital twin representing {name}.

A reply is acceptable when it:
- answers the user's question clearly and professionally;
- is faithful to the supplied personal information;
- does not invent career facts;
- admits when the available information is insufficient.

Use this reference information:

## Summary
{summary}

## LinkedIn profile
{linkedin}

Return whether the reply is acceptable and concise feedback."""

In [ ]:
def evaluator_user_prompt(reply, message, history):
    return f"""Conversation history:
{history}

Latest user message:
{message}

Digital twin reply:
{reply}

Evaluate only the final reply and provide actionable feedback if it is unacceptable."""


def evaluate_reply(reply, message, history):
    messages = [
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user", "content": evaluator_user_prompt(reply, message, history)},
    ]
    response = gemini.beta.chat.completions.parse(
        model="gemini-2.5-flash",
        messages=messages,
        response_format=Evaluation,
    )
    return response.choices[0].message.parsed


def revise_reply(reply, message, history, feedback):
    revised_system_prompt = system_prompt + f"""

## Previous reply rejected
Your previous reply was:
{reply}

The evaluator gave this feedback:
{feedback}

Write a corrected reply. Do not mention the evaluation process."""

    messages = (
        [{"role": "system", "content": revised_system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )
    return response.choices[0].message.content

In [ ]:
def chat(message, history):
    user_message = message
    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": user_message}]
    )

    while True:
        response = openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
        )

        if response.choices[0].finish_reason == "tool_calls":
            assistant_message = response.choices[0].message
            results = handle_tool_calls(assistant_message.tool_calls)
            messages.append(assistant_message)
            messages.extend(results)
        else:
            reply = response.choices[0].message.content
            break

    evaluation = evaluate_reply(reply, user_message, history)

    if evaluation.is_acceptable:
        print("Passed evaluation")
        return reply

    print("Failed evaluation, revising reply")
    print(evaluation.feedback)
    return revise_reply(
        reply,
        user_message,
        history,
        evaluation.feedback,
    )


# Launch this interface after the original app setup cells have run.
gr.ChatInterface(chat, type="messages").launch()